# Exp 01: First overview

Card: [exp01_first_overview.md](../../experiments/exp01_first_overview.md)


In [ ]:
import itertools
import os

import matplotlib.pyplot as plt
import mlflow
import pandas as pd
from PIL import Image

from core.config import CLASSES, ROOT, SEED, TEST_DIR, TRAIN_DIR
from core.data import read_test
from core.tracking import Experiment

SMOKE = os.environ.get("SMOKE") == "1"
if SMOKE:
    N_IMAGES = 20  # header reads per split
    N_SHEET = 4  # images per contact sheet
else:
    N_IMAGES = None  # all images
    N_SHEET = 16

EXPERIMENT = "cnn-eda-first-overview" + ("-smoke" if SMOKE else "")
FIGURES = ROOT / "sections" / "eda" / "figures"
SAVE_FIGURES = not SMOKE

# dataviz reference palette, light mode
SURFACE, INK, INK_2, MUTED, GRID, SERIES = (
    "#fcfcfb",
    "#0b0b0b",
    "#52514e",
    "#898781",
    "#e1e0d9",
    "#2a78d6",
)

run = Experiment(EXPERIMENT).run("exp01")

## Data


In [ ]:
train = pd.read_csv(TRAIN_DIR.parent / "train.csv")  # raw labels, no folds needed
test = read_test()
if SMOKE:
    train = train.sample(N_IMAGES * 10, random_state=SEED)
    test = test.head(N_IMAGES)
len(train), len(test)

## Image sizes and formats


In [ ]:
def header(path):
    # Image.open reads header only, pixels stay undecoded
    with Image.open(path) as im:
        return im.width, im.height, im.format, im.mode


def size_table(df, img_dir):
    paths = [img_dir / f"{i}.jpg" for i in df["image_id"].head(N_IMAGES)]
    info = pd.DataFrame(
        [header(p) for p in paths], columns=["width", "height", "format", "mode"]
    )
    t = info.value_counts().rename("count").reset_index()
    t["ratio"] = (t["count"] / len(info)).round(4)
    return t


train_sizes = size_table(train, TRAIN_DIR)
test_sizes = size_table(test, TEST_DIR)
mlflow.log_table(train_sizes, "train_sizes.json")
mlflow.log_table(test_sizes, "test_sizes.json")
train_sizes

In [ ]:
test_sizes

## Class ratio


In [ ]:
ratio = train[CLASSES].mean()
ratio["no cats or dogs"] = (train[CLASSES].sum(axis=1) == 0).mean()
mlflow.log_metrics({f"ratio_{c.replace(' ', '_')}": v for c, v in ratio.items()})

# groups split by gaps: generic classes, VIP animals, none; "no cats or dogs" gray, not a class
bar_groups = [["Cat", "Dog"], ["Moris", "Motya", "Biatrix"], ["no cats or dogs"]]
labels, ys, y = [], [], 0.0
for g in bar_groups:
    for c in g:
        labels.append(c)
        ys.append(-y)
        y += 1
    y += 0.6
colors = [MUTED if c == "no cats or dogs" else SERIES for c in labels]

fig, ax = plt.subplots(figsize=(7, 3.6), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
ax.barh(ys, ratio[labels].values, height=0.62, color=colors)
for yi, c in zip(ys, labels):
    n = int(round(ratio[c] * len(train)))
    ax.annotate(
        f"{ratio[c]:.1%}",
        (ratio[c], yi),
        xytext=(5, 0),
        textcoords="offset points",
        va="center",
        color=INK,
        fontsize=10,
        fontweight="bold",
    )
    ax.annotate(
        f"{n} images",
        (ratio[c], yi),
        xytext=(48, 0),
        textcoords="offset points",
        va="center",
        color=INK_2,
        fontsize=9,
    )
ax.set_yticks(ys, labels)
ax.tick_params(length=0, labelsize=10, labelcolor=INK)
ax.set_xlim(0, ratio.max() * 1.3)
ax.set_xticks([])  # direct labels carry values, axis dropped
for side in ax.spines:
    ax.spines[side].set_visible(False)
ax.axvline(0, color="#c3c2b7", linewidth=1)
fig.text(
    0.02,
    0.95,
    "Share of train images per class",
    color=INK,
    fontsize=12,
    fontweight="bold",
)
fig.text(
    0.02,
    0.89,
    f"multilabel, shares do not sum to 100%; n = {len(train)}",
    color=INK_2,
    fontsize=9,
)
fig.tight_layout(rect=(0, 0, 1, 0.87))
if SAVE_FIGURES:
    FIGURES.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURES / "exp01_first_overview_classes.png", dpi=150)
    mlflow.log_figure(fig, "exp01_first_overview_classes.png")

## Label combinations


In [ ]:
combos = pd.DataFrame(
    list(itertools.product([0, 1], repeat=len(CLASSES))), columns=CLASSES
)
counts = train.groupby(CLASSES).size().rename("count").reset_index()
combos = (
    combos.merge(counts, on=CLASSES, how="left")
    .fillna({"count": 0})
    .astype({"count": int})
)
combos["ratio"] = (combos["count"] / len(train)).round(4)
mlflow.log_table(combos, "label_combinations.json")
combos

## Contact sheets


In [ ]:
groups = {c: train[c] == 1 for c in CLASSES}
groups["no cats or dogs"] = train[CLASSES].sum(axis=1) == 0
groups["cat and dog"] = (train["Cat"] == 1) & (train["Dog"] == 1)


def contact_sheet(name, mask):
    ids = train.loc[mask, "image_id"]
    sample = ids.sample(min(N_SHEET, len(ids)), random_state=SEED)
    cols = 8
    rows = max(1, -(-len(sample) // cols))
    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(cols * 1.25, rows * 1.25 + 0.4),
        squeeze=False,
        facecolor=SURFACE,
    )
    for ax in axes.flat:
        ax.axis("off")
    for ax, i in zip(axes.flat, sample):
        im = Image.open(TRAIN_DIR / f"{i}.jpg")
        ax.imshow(im.resize((im.width // 2, im.height // 2)))
    fig.suptitle(f"{name} (n = {len(ids)})", x=0.01, ha="left", color=INK)
    fig.tight_layout()
    if SAVE_FIGURES:
        slug = name.replace(" ", "_").lower()
        fig.savefig(FIGURES / f"exp01_first_overview_sheet_{slug}.png", dpi=100)
        mlflow.log_figure(fig, f"exp01_first_overview_sheet_{slug}.png")
    plt.show()


for name, mask in groups.items():
    contact_sheet(name, mask)

In [ ]:
run.end()